In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)

df = pd.read_csv("data/raw/viral_videos.csv")
print(df.shape)
df.head()

(12000, 15)


,video_id,category,duration_sec,title_length,thumbnail_face,upload_hour,upload_day,tags_count,description_length,subscriber_count,views,likes,comments,shares,viral
0,vid_000001,Beauty,517,64,1,23,Wed,27,429,907474,970600,20610,4029,7208,1
1,vid_000002,Tech,495,47,1,23,Mon,10,222,1004641,104278,3907,554,554,0
2,vid_000003,Beauty,311,76,0,17,Mon,13,520,168170,18102,1021,77,122,0
3,vid_000004,Fitness,590,69,0,0,Mon,17,309,285593,52042,2587,142,964,0
4,vid_000005,Tech,380,58,1,19,Mon,0,101,1073326,908509,22993,1825,2611,1


In [2]:
df.info(),df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   video_id            12000 non-null  str  
 1   category            12000 non-null  str  
 2   duration_sec        12000 non-null  int64
 3   title_length        12000 non-null  int64
 4   thumbnail_face      12000 non-null  int64
 5   upload_hour         12000 non-null  int64
 6   upload_day          12000 non-null  str  
 7   tags_count          12000 non-null  int64
 8   description_length  12000 non-null  int64
 9   subscriber_count    12000 non-null  int64
 10  views               12000 non-null  int64
 11  likes               12000 non-null  int64
 12  comments            12000 non-null  int64
 13  shares              12000 non-null  int64
 14  viral               12000 non-null  int64
dtypes: int64(12), str(3)
memory usage: 1.6 MB


(None,
 video_id              0
 category              0
 duration_sec          0
 title_length          0
 thumbnail_face        0
 upload_hour           0
 upload_day            0
 tags_count            0
 description_length    0
 subscriber_count      0
 views                 0
 likes                 0
 comments              0
 shares                0
 viral                 0
 dtype: int64)

In [3]:
for col in ["category", "upload_day"]:
    print(f"\n{col}: {df[col].nunique()} unique")
    print(df[col].value_counts(dropna=False))


category: 15 unique
category
News         825
Tech         824
Finance      821
Vlog         821
Beauty       817
Science      808
Travel       807
Gaming       807
Sports       806
Cooking      794
DIY          779
Fitness      776
Music        776
Comedy       775
Education    764
Name: count, dtype: int64

upload_day: 7 unique
upload_day
Mon    1751
Sun    1746
Wed    1743
Thu    1728
Sat    1708
Fri    1697
Tue    1627
Name: count, dtype: int64


In [4]:
df[["duration_sec","title_length","upload_hour","tags_count",
    "description_length","subscriber_count","views","likes",
    "comments","shares"]].describe().T

,count,mean,std,min,25%,50%,75%,max
duration_sec,12000.0,6.316196e+02,2.858254e+02,30.0,408.00,614.0,822.00,1960.0
title_length,12000.0,5.452400e+01,1.495079e+01,10.0,45.00,55.0,65.00,109.0
upload_hour,12000.0,1.153942e+01,6.914424e+00,0.0,5.00,12.0,18.00,23.0
tags_count,12000.0,1.458683e+01,7.741422e+00,0.0,9.00,14.0,20.00,44.0
description_length,12000.0,5.089194e+02,2.872129e+02,0.0,300.00,504.0,709.00,1618.0
subscriber_count,12000.0,1.009487e+06,2.585562e+06,3636.0,158521.50,381382.0,929844.75,100000000.0
views,12000.0,7.677524e+05,3.526908e+06,139.0,61235.50,185268.0,556970.50,199465656.0
likes,12000.0,3.750578e+04,1.823115e+05,6.0,2790.00,8751.5,26571.25,11592871.0
comments,12000.0,4.739626e+03,2.331387e+04,0.0,316.75,1018.0,3207.25,1116186.0
shares,12000.0,9.503240e+03,4.644499e+04,1.0,641.75,2050.5,6527.25,2367174.0


In [5]:
print(df["viral"].value_counts(dropna=False))
print(f"viral rate: {df['viral'].mean():.3f}")

viral
0    9600
1    2400
Name: count, dtype: int64
viral rate: 0.200


In [6]:
df_clean = df.copy()

# 1. normalize string categories: lowercase + strip
for col in ["category", "upload_day"]:
    df_clean[col] = (df_clean[col]
                     .astype(str)
                     .str.strip()
                     .str.lower())

# 2. map 3-letter days -> consistent 3-letter lowercase
#    (if your data is already "Mon", "Tue"... this is a no-op, but makes it robust)
day_map = {
    "monday":"mon","tuesday":"tue","wednesday":"wed","thursday":"thu",
    "friday":"fri","saturday":"sat","sunday":"sun",
    "mon":"mon","tue":"tue","wed":"wed","thu":"thu",
    "fri":"fri","sat":"sat","sun":"sun",
}
df_clean["upload_day"] = df_clean["upload_day"].map(day_map).fillna("unknown")

# 3. coerce numeric columns (bad strings -> NaN)
num_cols = ["duration_sec","title_length","upload_hour","tags_count",
            "description_length","subscriber_count","views","likes",
            "comments","shares"]
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# 4. clip absurd values
df_clean["duration_sec"] = df_clean["duration_sec"].clip(lower=0)
df_clean["subscriber_count"] = df_clean["subscriber_count"].clip(lower=0)
df_clean["upload_hour"] = df_clean["upload_hour"].clip(lower=0, upper=23)
df_clean["title_length"] = df_clean["title_length"].clip(lower=0)
df_clean["tags_count"] = df_clean["tags_count"].clip(lower=0)
df_clean["description_length"] = df_clean["description_length"].clip(lower=0)

# 5. thumbnail_face: force 0/1 int
df_clean["thumbnail_face"] = (
    pd.to_numeric(df_clean["thumbnail_face"], errors="coerce")
      .fillna(0).astype(int).clip(0, 1)
)

# 6. fill small NaN counts in numeric with median
for col in num_cols:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 7. drop rows missing the target — can't train on those
before = len(df_clean)
df_clean = df_clean.dropna(subset=["viral"])
df_clean["viral"] = df_clean["viral"].astype(int)
print(f"dropped {before - len(df_clean)} rows without target")

# 8. drop duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"dropped {before - len(df_clean)} duplicate rows")

print("\nclean shape:", df_clean.shape)

dropped 0 rows without target
dropped 0 duplicate rows

clean shape: (12000, 15)


In [9]:
print("NaN after clean:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])
print("\ncategories:", sorted(df_clean["category"].unique()))
print("days:", sorted(df_clean["upload_day"].unique()))
print("\nviral rate:", round(df_clean["viral"].mean(), 3))
df_clean.describe().T

NaN after clean:
Series([], dtype: int64)

categories: ['beauty', 'comedy', 'cooking', 'diy', 'education', 'finance', 'fitness', 'gaming', 'music', 'news', 'science', 'sports', 'tech', 'travel', 'vlog']
days: ['fri', 'mon', 'sat', 'sun', 'thu', 'tue', 'wed']

viral rate: 0.2


,count,mean,std,min,25%,50%,75%,max
duration_sec,12000.0,6.316196e+02,2.858254e+02,30.0,408.00,614.0,822.00,1960.0
title_length,12000.0,5.452400e+01,1.495079e+01,10.0,45.00,55.0,65.00,109.0
thumbnail_face,12000.0,4.002500e-01,4.899693e-01,0.0,0.00,0.0,1.00,1.0
upload_hour,12000.0,1.153942e+01,6.914424e+00,0.0,5.00,12.0,18.00,23.0
tags_count,12000.0,1.458683e+01,7.741422e+00,0.0,9.00,14.0,20.00,44.0
description_length,12000.0,5.089194e+02,2.872129e+02,0.0,300.00,504.0,709.00,1618.0
subscriber_count,12000.0,1.009487e+06,2.585562e+06,3636.0,158521.50,381382.0,929844.75,100000000.0
views,12000.0,7.677524e+05,3.526908e+06,139.0,61235.50,185268.0,556970.50,199465656.0
likes,12000.0,3.750578e+04,1.823115e+05,6.0,2790.00,8751.5,26571.25,11592871.0
comments,12000.0,4.739626e+03,2.331387e+04,0.0,316.75,1018.0,3207.25,1116186.0


In [10]:
df_clean.to_csv("data/raw/viral_videos_clean.csv", index=False)
print("saved -> data/raw/viral_videos_clean.csv")

saved -> data/raw/viral_videos_clean.csv


In [11]:
corr = df_clean.select_dtypes(include=[np.number]).corr()["viral"].sort_values(ascending=False)
corr

viral                 1.000000
subscriber_count      0.413938
views                 0.329018
likes                 0.307969
shares                0.306977
comments              0.305907
thumbnail_face        0.056724
tags_count            0.029028
upload_hour           0.014626
title_length          0.000674
description_length   -0.011695
duration_sec         -0.156353
Name: viral, dtype: float64